# 06b — InternViT-6B target (SAMGA backbone): layer sweep, centred routing, 3 seeds, LOSO 10 folds

Notebook 06 ran with the DINOv2-giant fallback (InternViT failed on the `flash_attn` import check). Results to beat (DINOv2, seed 0):
within L20 70.5 · route 66.3 | LOSO route plain 21.6 · CSLS 31.2 · recovery 35.3.

| Step | What |
|---|---|
| Cell 2 | InternViT-6B-448px-V2.5 features, layers 12–36 (mean + CLS), `flash_attn` check patched, `use_flash_attn=False`; saved in chunks (a disconnect resumes) |
| Cell 5 | within-subject single-layer sweep (seed 0) → best layer by **validation** → `route_c` = 5 layers centred on it → seeds 1, 2 for best single and `route_c` |
| Cell 6 | LOSO 10 folds, 63 channels, `route_c` and best single; plain / CSLS / SCORE-style recovery (transductive, label-free) |
| Cell 7 | summary incl. notebook-06 DINOv2 rows and paper numbers |

Same EEG cache, image list, validation split, model, training recipe (60 ep, AdamW, batch 1024, lr 1e-4, wd 1e-4) and scoring as notebook 06. Outputs: `eeg_results/v6b`.

**GPU:** A100 for cell 2 (~30–45 min once). L4 works (~1.5 h, resumable). Cells 5–6 on L4: within ~0.5 min per run, LOSO ~1 min per fold.
**If cell 2 fails with a transformers error inside the InternViT remote code:** run `!pip install -q transformers==4.45.2`, restart the runtime, run again (parts already saved are kept).

In [ ]:
# 0. Setup
import os, sys, gc, json, math, time, random, shutil, glob, copy, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")
DCACHE = f"{DRIVE}/cache"
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")
OUT    = f"{DRIVE}/v6b"
OUT6   = f"{DRIVE}/v6"                                  # notebook-06 results (DINOv2), for the summary
for d in [DRIVE, DCACHE, ROOT, LOCAL, OUT]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
SMOKE = os.environ.get("EEG_SMOKE") == "1"           # tiny synthetic run for code checking only

import numpy as np
for pkg, pip_name in [("huggingface_hub", "huggingface_hub"), ("pandas", "pandas"), ("transformers", "transformers"),
                      ("timm", "timm"), ("einops", "einops")]:
    if not SMOKE and importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8      # bfloat16 needs Ampere or newer (not T4)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| bf16:", AMP, "| torch", torch.__version__)
print(f"Drive free: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB (features need ~1.2 GB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

In [ ]:
# 1. Image list (same order as notebooks 04/05) and the validation split
META = f"{DCACHE}/meta_v6.pt"
if not os.path.exists(META):
    import tarfile
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    torch.save({"train_files": train_files, "test_files": test_files,
                "train_rel": [os.path.relpath(idx_train[f], ROOT) for f in train_files],
                "test_rel": [os.path.relpath(idx_test[f], ROOT) for f in test_files],
                "n_concepts": len(set(meta["train_img_concepts"]))}, META)
M = torch.load(META, weights_only=False)
N_TR, N_TE = len(M["train_files"]), len(M["test_files"])
PER = N_TR // M["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)              # last image of every training concept (same as 01–05)
print(f"train images {N_TR} | test images {N_TE} | concepts {M['n_concepts']} | val images {IS_VAL.sum()}")

In [ ]:
# 2. InternViT-6B intermediate-layer features, cached on Drive. Extracted in chunks so a disconnect resumes.
REPO   = "OpenGVLab/InternViT-6B-448px-V2_5"
LAYERS = [12, 16, 20, 24, 28, 32, 36]            # SAMGA uses 20–36; 12/16 come for free in the same forward pass
BACKBONE = "internvit6b"
FEAT  = f"{DCACHE}/feat6_{BACKBONE}.pt"
PARTS = f"{DCACHE}/feat6_{BACKBONE}_parts"; os.makedirs(PARTS, exist_ok=True)
CHUNK = 2048

def load_internvit():
    # The remote code imports flash_attn inside a try block, but transformers' import check rejects it before running.
    import transformers.dynamic_module_utils as dmu
    from transformers import AutoModel, CLIPImageProcessor, PreTrainedModel
    if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
        PreTrainedModel.all_tied_weights_keys = {}      # transformers 5 expects it; InternViT remote code (v4) never sets it
    if not getattr(dmu, "_patched_flash", False):
        _orig = dmu.get_imports
        dmu.get_imports = lambda f: [m for m in _orig(f) if m != "flash_attn"]
        dmu._patched_flash = True
    dt = torch.bfloat16 if AMP else torch.float16
    proc = CLIPImageProcessor.from_pretrained(REPO)
    model = AutoModel.from_pretrained(REPO, torch_dtype=dt, low_cpu_mem_usage=True, trust_remote_code=True,
                                      use_flash_attn=False)
    model.encoder.layers = model.encoder.layers[:max(LAYERS)]           # deeper blocks are never used
    return proc, model.to(device).eval(), dt

def extract():
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    t0 = time.time()
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        import tarfile
        from huggingface_hub import hf_hub_download
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    proc, model, dt = load_internvit()
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == "cuda" else 0
    BS = 16 if gb > 35 else 8
    print(f"InternViT-6B loaded in {(time.time()-t0)/60:.1f} min | GPU {gb:.0f} GB | batch {BS}", flush=True)

    class DS(Dataset):
        def __init__(self, rel): self.rel = rel
        def __len__(self): return len(self.rel)
        def __getitem__(self, i): return Image.open(f"{ROOT}/{self.rel[i]}").convert("RGB")
    collate = lambda ims: proc(images=ims, return_tensors="pt")["pixel_values"]

    def run(rel, tag):
        n_chunks = math.ceil(len(rel) / CHUNK)
        for c in range(n_chunks):
            p = f"{PARTS}/{tag}_{c:03d}.pt"
            if os.path.exists(p): continue
            out = {(l, q): [] for l in LAYERS for q in ("mean", "cls")}
            for px in DataLoader(DS(rel[c * CHUNK:(c + 1) * CHUNK]), batch_size=BS, num_workers=4, collate_fn=collate):
                with torch.no_grad():
                    hs = model(pixel_values=px.to(device, dt), output_hidden_states=True).hidden_states
                for l in LAYERS:
                    h = hs[l].float()
                    out[(l, "cls")].append(h[:, 0].cpu().half()); out[(l, "mean")].append(h[:, 1:].mean(1).cpu().half())
            torch.save({k: torch.cat(v) for k, v in out.items()}, p + ".tmp"); os.replace(p + ".tmp", p)
            print(f"  {tag} chunk {c+1}/{n_chunks} | {(time.time()-t0)/60:.1f} min", flush=True)
        parts = [torch.load(f"{PARTS}/{tag}_{c:03d}.pt") for c in range(n_chunks)]
        return {k: torch.cat([q[k] for q in parts]) for k in parts[0]}
    tr, te = run(M["train_rel"], "train"), run(M["test_rel"], "test")
    assert len(tr[(LAYERS[0], "mean")]) == N_TR and len(te[(LAYERS[0], "mean")]) == N_TE
    torch.save({"layers": LAYERS, "tr": tr, "te": te}, FEAT + ".tmp"); os.replace(FEAT + ".tmp", FEAT)
    del model; free_gpu()
    print(f"features done in {(time.time()-t0)/60:.1f} min -> {FEAT}")

if not os.path.exists(FEAT): extract()          # no silent fallback: DINOv2 results already exist in v6
FE = torch.load(FEAT, weights_only=False)
LAYERS = FE["layers"]
print("backbone:", BACKBONE, "| layers:", LAYERS, "| dim:", FE["tr"][(LAYERS[0], "mean")].shape[1])
F04 = f"{DCACHE}/feats3_ViT-H-14_laion2b_s32b_b79k.pt"
H14 = None                                       # old CLIP target already compared in notebook 06


In [ ]:
# 3. Targets: per-dimension z-score with training statistics (intermediate ViT layers have a few huge-norm dims)
def target(name, pool="mean"):
    if name == "h14final":
        tr, te = H14["img_tr"].float(), H14["img_te"].float()
    else:
        tr, te = FE["tr"][(name, pool)].float(), FE["te"][(name, pool)].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((tr - m) / s).to(device), ((te - m) / s).to(device)

# EEG channels. THINGS-EEG2 preprocessing order (Gifford et al. 2022); the 17 posterior channels used by NICE/UBP/SAMGA.
CH63 = ["Fp1","Fp2","AF7","AF3","AFz","AF4","AF8","F7","F5","F3","F1","F2","F4","F6","F8","FT9","FT7","FC5","FC3","FC1",
        "FCz","FC2","FC4","FC6","FT8","FT10","T7","C5","C3","C1","Cz","C2","C4","C6","T8","TP9","TP7","CP5","CP3","CP1",
        "CPz","CP2","CP4","CP6","TP8","TP10","P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8",
        "O1","Oz","O2"]
POST17 = ["P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8","O1","Oz","O2"]
assert len(CH63) == 63
_, N_CH_ALL, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
CH_IDX = {"all": list(range(N_CH_ALL)),
          "post17": [CH63.index(c) for c in POST17] if N_CH_ALL == 63 else list(range(N_CH_ALL))}
print("EEG:", N_CH_ALL, "channels x", N_T, "samples | post17 indices", CH_IDX["post17"][:3], "...")

def load_eeg(sub, split, ch):
    x = np.load(local_copy(f"{sub}_{split}_z16.npy"), mmap_mode="r")
    return torch.from_numpy(np.ascontiguousarray(x[:, CH_IDX[ch]])).to(device)

VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device); TRN = torch.from_numpy(np.where(~IS_VAL)[0]).to(device)

In [ ]:
# 4. Model: EEGProject (MLP with residual bottleneck, as in Shallow Alignment) + linear image projector(s)
#    Single-layer target, or a learned softmax routing over several layers (global SAMGA-style prior, no subject residual).
EMB = 1024

class EEGProject(nn.Module):
    def __init__(self, n_in, d=EMB, drop=0.3):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d, d), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, x):
        h = self.inp(x.flatten(1)); return self.norm(h + self.res(h))

class ImageSide(nn.Module):
    def __init__(self, dims, init_logits=None):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, EMB) for d in dims])
        self.logits = nn.Parameter(torch.tensor(init_logits, dtype=torch.float) if init_logits is not None else torch.zeros(len(dims)),
                                   requires_grad=len(dims) > 1)
    def weights(self): return F.softmax(self.logits, 0)
    def forward(self, feats):                   # feats: list of (N, d_l), same rows
        w = self.weights()
        return sum(w[i] * p(f) for i, (p, f) in enumerate(zip(self.proj, feats)))

class Aligner(nn.Module):
    def __init__(self, n_in, dims, init_logits=None):
        super().__init__()
        self.eeg, self.img = EEGProject(n_in), ImageSide(dims, init_logits)
        self.log_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    def scale(self): return self.log_scale.exp().clamp(max=100)

def clip_loss(ze, zi, scale):
    ze, zi = F.normalize(ze, dim=-1), F.normalize(zi, dim=-1)
    logits = scale * ze @ zi.T; y = torch.arange(len(ze), device=ze.device)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

@torch.no_grad()
def embed_eeg(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
            out.append(F.normalize(model.eeg(X[i:i + bs].float()).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_img(model, feats):
    model.eval(); return F.normalize(model.img(feats).float(), dim=-1)

@torch.no_grad()
def metrics(sim):
    # sim: queries x candidates, the correct candidate of query i is candidate i
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

EPOCHS, BATCH, LR, WD, EVAL_EVERY = (60, 1024, 1e-4, 1e-4, 5) if not SMOKE else (3, 64, 1e-3, 1e-4, 1)
ROUTE_INIT = [-2.0, -1.0, 0.0, -1.0, -2.0]

def train(X, img_of, T_tr, val_sets, seed=0, log=None, init_logits=None):
    # X: EEG trials, img_of[i]: training image of trial i, T_tr: list of target matrices (N_TR x d_l)
    # val_sets: list of (EEG rows, image rows) used for model selection (held-out training images)
    set_seed(seed)
    model = Aligner(X[0].numel(), [t.shape[1] for t in T_tr], init_logits).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    tr_rows = torch.where(~torch.from_numpy(IS_VAL).to(device)[img_of])[0]
    steps = EPOCHS * math.ceil(len(tr_rows) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_rows[torch.randperm(len(tr_rows), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            imgs = img_of[b]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
                ze = model.eeg(X[b].float()); zi = model.img([t[imgs] for t in T_tr])
            loss = clip_loss(ze.float(), zi.float(), model.scale())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            zi_val = embed_img(model, [t[VAL] for t in T_tr])
            va = float(np.mean([metrics(embed_eeg(model, X[er]) @ zi_val[ir].T)["top1"] for er, ir in val_sets]))
            if log: log(ep, va, model)
            if va > best: best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_ep, best

In [ ]:
# 5. Within-subject, 10 subjects, 17 posterior channels.
#    (a) single-layer sweep, seed 0  -> best layer by mean VALIDATION top-1 (test never used for selection)
#    (b) route_c: 5 consecutive layers centred on the best layer, logits init -|distance| (SAMGA prior, recentred)
#    (c) seeds 1, 2 for the best single layer and route_c. CSLS on the test gallery is reported too (transductive).
W_SUBJECTS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
SEEDS = [0, 1, 2] if not SMOKE else [0, 1]

def get_targets(layers, pool):
    tt = [target(l, pool) for l in layers]
    return [a for a, _ in tt], [b for _, b in tt]

@torch.no_grad()
def csls(S, k=10):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

def run_within(cname, layers, pool, seeds, init=None, ch="post17"):
    T_tr, T_te = get_targets(layers, pool)
    for seed in seeds:
        for sub in W_SUBJECTS:
            path = f"{OUT}/within_{cname}_{sub}_s{seed}.json"
            if os.path.exists(path): continue
            t0 = time.time()
            X, Xt = load_eeg(sub, "train", ch), load_eeg(sub, "test", ch)
            model, best_ep, best_val = train(X, torch.arange(N_TR, device=device), T_tr,
                                             [(VAL, torch.arange(len(VAL), device=device))], seed, init_logits=init)
            sim = embed_eeg(model, Xt) @ embed_img(model, T_te).T
            r = dict(config=cname, layers=[str(l) for l in layers], pool=pool, channels=ch, subject=sub, seed=seed,
                     best_epoch=best_ep, val_top1=best_val, **metrics(sim),
                     **{f"csls_{k}": v for k, v in metrics(csls(sim)).items()},
                     route_w=[round(float(w), 3) for w in model.img.weights().detach()], minutes=round((time.time() - t0) / 60, 2))
            save_json(r, path)
            print(f"{cname:9s} {sub} s{seed}: top1 {r['top1']:5.1f} (CSLS {r['csls_top1']:5.1f}) top5 {r['top5']:5.1f} "
                  f"| val {best_val:5.1f} ep {best_ep} | w {r['route_w']} ({r['minutes']} min)", flush=True)
            del X, Xt, model; free_gpu()
    del T_tr, T_te; free_gpu()

def load_W():
    return pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/within_*.json"))])

# (a) sweep
for l in LAYERS: run_within(f"L{l}", [l], "mean", [0])
W = load_W()
sweep = W[W.config.str.match(r"^L\d+$") & (W.seed == 0)].groupby("config")[["val_top1", "top1"]].mean()
print(sweep.assign(layer=[int(c[1:]) for c in sweep.index]).sort_values("layer").round(1).to_string())
BEST_SINGLE = sweep.val_top1.idxmax(); b = LAYERS.index(int(BEST_SINGLE[1:]))
lo = min(max(0, b - 2), max(0, len(LAYERS) - 5)); ROUTE_LAYERS = LAYERS[lo:lo + 5]
ROUTE_C_INIT = [-abs(i - (b - lo)) * 1.0 for i in range(len(ROUTE_LAYERS))]
print(f"best single layer by validation: {BEST_SINGLE} | route_c layers {ROUTE_LAYERS} init {ROUTE_C_INIT}")

# (b) + (c)
run_within("route_c", ROUTE_LAYERS, "mean", SEEDS, init=ROUTE_C_INIT)
run_within(BEST_SINGLE, [int(BEST_SINGLE[1:])], "mean", SEEDS[1:])
run_within(f"{BEST_SINGLE}cls", [int(BEST_SINGLE[1:])], "cls", [0])

W = load_W(); W.to_csv(f"{OUT}/results_within_v6b.csv", index=False)
print(W.groupby("config")[["top1", "csls_top1", "top5", "val_top1"]].mean().round(1).sort_values("val_top1", ascending=False).to_string())


In [ ]:
# 6. Leave-one-subject-out, 10 folds, 63 channels: train on 9 subjects, test on the unseen one (no labels of the new person).
#    Targets: route_c and the best single layer (both chosen on validation in cell 5). Seed 0.
#    Scoring on the held-out person: plain cosine | CSLS | SCORE-style coordinate recovery (transductive, label-free).
X_CONFIGS = [("route_c", ROUTE_LAYERS, ROUTE_C_INIT), (BEST_SINGLE, [int(BEST_SINGLE[1:])], None)]
X_FOLDS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
print("LOSO targets:", X_CONFIGS)

CSLS_K, N_LAND, RHO = 10, 64, 0.1

@torch.no_grad()
def csls(S, k=CSLS_K):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

@torch.no_grad()
def recovery(Z, T, n_land=N_LAND, rho=RHO):
    # SCORE (arXiv 2608.19134) coordinate recovery, one step: moment matching, CSLS, mutual-nearest-neighbour landmarks
    # weighted by their margin, weighted orthogonal Procrustes with identity regularisation, then CSLS scoring.
    Zm = F.normalize((Z - Z.mean(0)) / (Z.std(0) + 1e-6) * T.std(0) + T.mean(0), dim=-1)
    S = csls(Zm @ T.T)
    top2 = S.topk(2, dim=1); j = top2.indices[:, 0]; i = torch.arange(len(S), device=S.device)
    mnn = S.argmax(0)[j] == i
    margin = (top2.values[:, 0] - top2.values[:, 1]).clamp(min=1e-4)
    cand = torch.where(mnn)[0]
    if len(cand) < 3: return S, 0, None
    keep = cand[margin[cand].topk(min(n_land, len(cand))).indices]
    w = margin[keep]; w = len(keep) * w / w.sum()
    Xl, Yl = Zm[keep], T[j[keep]]
    wn = w / w.sum(); muX = (wn[:, None] * Xl).sum(0); muY = (wn[:, None] * Yl).sum(0)
    C = ((Xl - muX) * w[:, None]).T @ (Yl - muY)
    Mx = C + rho * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device)
    U, _, Vh = torch.linalg.svd(Mx.double()); R = (U @ Vh).float()
    prec = float((keep == j[keep]).float().mean() * 100)          # diagnostic only (uses the answer key, never used for fitting)
    return csls(F.normalize((Zm - muX) @ R + muY, dim=-1) @ T.T), len(keep), prec

for cname, layers, init in X_CONFIGS:
    T_tr, T_te = get_targets(layers, "mean")
    for sub in X_FOLDS:
        path = f"{OUT}/loso_{cname}_{sub}.json"
        if os.path.exists(path): print(f"{cname} {sub}: done"); continue
        t0 = time.time(); srcs = [s for s in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:3]) if s != sub]
        X = torch.empty((len(srcs) * N_TR, N_CH_ALL, N_T), dtype=torch.float16, device=device)
        for k, s in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = load_eeg(s, "train", "all")
        img_of = torch.arange(len(X), device=device) % N_TR
        vsets = [(k * N_TR + VAL, torch.arange(len(VAL), device=device)) for k in range(min(3, len(srcs)))]
        log = lambda ep, va, m: print(f"  {cname} {sub} ep {ep:2d} | val top1 {va:.1f} | {(time.time()-t0)/60:.1f} min", flush=True) if ep % 10 == 0 else None
        model, best_ep, best_val = train(X, img_of, T_tr, vsets, 0, log, init)
        del X; free_gpu()
        Xt = load_eeg(sub, "test", "all")
        Z, Ti = embed_eeg(model, Xt), embed_img(model, T_te)
        S0 = Z @ Ti.T; S_rec, n_l, prec = recovery(Z, Ti)
        r = dict(config=cname, test_subject=sub, best_epoch=best_ep, val_top1=best_val,
                 route_w=[round(float(w), 3) for w in model.img.weights().detach()], n_landmarks=n_l, landmark_precision=prec,
                 **{f"plain_{k}": v for k, v in metrics(S0).items()},
                 **{f"csls_{k}": v for k, v in metrics(csls(S0)).items()},
                 **{f"recovery_{k}": v for k, v in metrics(S_rec).items()},
                 minutes=round((time.time() - t0) / 60, 2))
        torch.save(model.state_dict(), f"{OUT}/loso_{cname}_{sub}.pt")
        save_json(r, path)
        print(f"{cname} {sub} ({r['minutes']} min): plain {r['plain_top1']:.1f} | CSLS {r['csls_top1']:.1f} | "
              f"recovery {r['recovery_top1']:.1f} (landmarks {n_l}, precision {prec})", flush=True)
        del Xt, model; free_gpu()
    del T_tr, T_te; free_gpu()

L = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/loso_*.json"))])
L.to_csv(f"{OUT}/results_loso_v6b.csv", index=False)

In [ ]:
# 7. Summary (paper numbers are self-reported under their own protocols: indicative, not like-for-like)
def ms(s): return f"{s.mean():.1f} ± {s.std():.1f} (n={len(s)})"

def per_subject(df, col):                       # average seeds within a subject first, then spread across subjects
    return df.groupby("subject")[col].mean()

print("WITHIN-SUBJECT Top-1, InternViT-6B (mean ± SD over subjects; seeds averaged per subject)")
for c, g in W.groupby("config"):
    seeds = sorted(g.seed.unique())
    seed_sd = g.groupby("seed").top1.mean().std() if len(seeds) > 1 else float("nan")
    print(f"  {c:10s} {ms(per_subject(g, 'top1'))} | CSLS {per_subject(g, 'csls_top1').mean():.1f} | top5 {per_subject(g, 'top5').mean():.1f}"
          f" | seeds {seeds}" + (f" (SD over seeds {seed_sd:.1f})" if len(seeds) > 1 else ""))

print("\nLOSO Top-1, InternViT-6B")
for c, g in L.groupby("config"):
    print(f"  {c:10s} plain {ms(g.plain_top1)} | CSLS {ms(g.csls_top1)} | recovery {ms(g.recovery_top1)} | landmark prec {g.landmark_precision.mean():.1f}")
print("\nPer subject (LOSO, route_c):")
print(L[L.config == "route_c"][["test_subject", "plain_top1", "csls_top1", "recovery_top1", "landmark_precision"]].round(1).to_string(index=False))

rows = [["ATM (NeurIPS 2024)", 27.1, 11.9, None], ["UBP (CVPR 2025)", 50.9, 12.4, None], ["NeuroBridge (2026)", 63.2, 19.0, None],
        ["Shallow Alignment (2026)", 82.6, 22.4, None], ["SAMGA (2026)", 91.3, 34.4, None], ["SAMGA + SCORE recovery (2026)", None, 26.2, 48.8]]
try:                                             # notebook 06, DINOv2-giant, seed 0
    W6 = pd.read_csv(f"{OUT6}/results_within_v6.csv"); L6 = pd.read_csv(f"{OUT6}/results_loso_v6.csv")
    rows += [["Ours 06, DINOv2-g L20 / LOSO route (seed 0)", round(W6[W6.config == "L20"].top1.mean(), 1),
              round(L6[L6.config == "route"].plain_top1.mean(), 1), round(L6[L6.config == "route"].recovery_top1.mean(), 1)]]
except FileNotFoundError:
    print("(notebook-06 CSVs not found in", OUT6, ")")
for c in ["route_c", BEST_SINGLE]:
    w, l = W[W.config == c], L[L.config == c]
    rows.append([f"Ours 06b, InternViT-6B {c} ({len(w.seed.unique())} seed(s) within, 1 seed LOSO)",
                 round(per_subject(w, "top1").mean(), 1) if len(w) else None,
                 round(l.plain_top1.mean(), 1) if len(l) else None, round(l.recovery_top1.mean(), 1) if len(l) else None])
REPORTED = pd.DataFrame(rows, columns=["method", "intra_top1", "loso_top1", "loso_label_free_top1"])
print("\n", REPORTED.to_string(index=False))
REPORTED.to_csv(f"{OUT}/comparison_v6b.csv", index=False)

# make sure every file written to Drive is uploaded before the runtime is closed
try:
    from google.colab import drive; drive.flush_and_unmount(); print("Drive flushed: all files uploaded")
except Exception as e: print("flush skipped:", e)
